In [1]:
!pip install torch_geometric rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 51.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import Sequential, Linear, BatchNorm1d, ReLU, BCEWithLogitsLoss
from torch.optim import AdamW
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, GATv2Conv, global_mean_pool
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.metrics import matthews_corrcoef
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier
from collections import defaultdict
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import Descriptors, MACCSkeys
from sklearn.linear_model import LogisticRegression
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ==========================================
# 1. DATA PROCESSING & DEEP FEATURE EXTRACTION
# ==========================================

def extract_tabular(df):
    """Extracts purely tabular data for the Level 1 Tree Models and Level 2 Meta-Model."""
    X_fp, X_desc = [], []
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    
    for smiles in df['SMILE']:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            X_fp.append(np.zeros(2048 + 167))
            X_desc.append(np.zeros(8))
            continue
            
        fp = list(mfpgen.GetFingerprint(mol))
        maccs = list(MACCSkeys.GenMACCSKeys(mol))
        desc = [
            Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol), Descriptors.FractionCSP3(mol), Descriptors.RingCount(mol)
        ]
        X_fp.append(fp + maccs)
        X_desc.append(desc)
        
    return np.nan_to_num(np.array(X_fp, dtype=np.float32)), np.nan_to_num(np.array(X_desc, dtype=np.float32))
    
def get_atom_features(atom):
    """Rich 2D topological atom features."""
    atomic_num = atom.GetAtomicNum()
    degree = atom.GetTotalDegree()
    formal_charge = atom.GetFormalCharge()
    num_hs = atom.GetTotalNumHs()
    is_aromatic = int(atom.GetIsAromatic())
    is_in_ring = int(atom.IsInRing())
    
    hyb = atom.GetHybridization()
    hyb_sp = int(hyb == Chem.HybridizationType.SP)
    hyb_sp2 = int(hyb == Chem.HybridizationType.SP2)
    hyb_sp3 = int(hyb == Chem.HybridizationType.SP3)
    
    # 9 Features total
    return [atomic_num, degree, formal_charge, num_hs, is_aromatic, is_in_ring, hyb_sp, hyb_sp2, hyb_sp3]

def get_bond_features(bond):
    """One-hot encoded bond types and topological properties."""
    bt = bond.GetBondType()
    bt_single = int(bt == Chem.BondType.SINGLE)
    bt_double = int(bt == Chem.BondType.DOUBLE)
    bt_triple = int(bt == Chem.BondType.TRIPLE)
    bt_aromatic = int(bt == Chem.BondType.AROMATIC)
    is_conjugated = int(bond.GetIsConjugated())
    is_in_ring = int(bond.IsInRing())
    
    # 6 Features total
    return [bt_single, bt_double, bt_triple, bt_aromatic, is_conjugated, is_in_ring]

def smiles_to_graph(smiles, labels=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: 
        mol = Chem.MolFromSmiles('C')
    
    node_features = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_features, dtype=torch.float)
    
    edge_indices, edge_features = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_indices += [[i, j], [j, i]]
        bond_feat = get_bond_features(bond)
        edge_features += [bond_feat, bond_feat]
        
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous() if edge_indices else torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_features, dtype=torch.float) if edge_features else torch.empty((0, 6), dtype=torch.float)
    
    # 1. Morgan Fingerprint (2048)
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fp = list(mfpgen.GetFingerprint(mol))
    
    # 2. MACCS Keys (167) - Highly critical for structural alerts
    maccs = list(MACCSkeys.GenMACCSKeys(mol))
    
    # 3. Explicit 2D Descriptors (8)
    desc = [
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.FractionCSP3(mol), Descriptors.RingCount(mol)
    ]
    
    # Combine: 2048 + 167 + 8 = 2223 global features
    global_features = torch.tensor(fp + maccs + desc, dtype=torch.float).unsqueeze(0)
    global_features = torch.nan_to_num(global_features, nan=0.0)
    
    y = torch.tensor([labels], dtype=torch.float) if labels is not None else None
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, fp=global_features)

class MoleculeDataset(Dataset):
    def __init__(self, dataframe, is_test=False):
        super().__init__()
        self.graphs = []
        for _, row in dataframe.iterrows():
            labels = None if is_test else row[['P1', 'P2', 'P3', 'P4', 'P5']].values.astype(float)
            graph = smiles_to_graph(row['SMILE'], labels) # Matching Kaggle dataset schema
            if graph is not None: self.graphs.append(graph)

    def len(self): return len(self.graphs)
    def get(self, idx): return self.graphs[idx]

In [4]:
# ==========================================
# 2. MODELS (JK-Networks & Multi-Label Output)
# ==========================================
class GINModel(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_dim=128, fp_dim=2223):
        super().__init__()
        # 3 Layers of Message Passing
        nn1 = Sequential(Linear(num_node_features, hidden_dim), BatchNorm1d(hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINEConv(nn1, edge_dim=num_edge_features)
        
        nn2 = Sequential(Linear(hidden_dim, hidden_dim), BatchNorm1d(hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINEConv(nn2, edge_dim=num_edge_features)
        
        nn3 = Sequential(Linear(hidden_dim, hidden_dim), BatchNorm1d(hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv3 = GINEConv(nn3, edge_dim=num_edge_features)
        
        # FC accepts: (3 layers * hidden_dim) + fp_dim
        jk_dim = hidden_dim * 3
        self.fc1 = Linear(jk_dim + fp_dim, hidden_dim * 2)
        self.drop = torch.nn.Dropout(0.5)
        self.fc2 = Linear(hidden_dim * 2, 5) 

    def forward(self, data):
        x, edge_index, edge_attr, batch, fp = data.x, data.edge_index, data.edge_attr, data.batch, data.fp
        
        x1 = F.relu(self.conv1(x, edge_index, edge_attr))
        x2 = F.relu(self.conv2(x1, edge_index, edge_attr))
        x3 = F.relu(self.conv3(x2, edge_index, edge_attr))
        
        # JUMPING KNOWLEDGE: Concatenate all layer representations
        x_jk = torch.cat([x1, x2, x3], dim=-1)
        
        # Global Pooling
        x_pool = global_mean_pool(x_jk, batch)
        
        # Combine with Global Fingerprints
        x_out = torch.cat([x_pool, fp.squeeze(1)], dim=1)
        
        x_out = self.drop(F.relu(self.fc1(x_out)))
        return self.fc2(x_out)

class GATModel(torch.nn.Module):
    def __init__(self, num_node_features, num_edge_features, hidden_dim=128, heads=4, fp_dim=2223):
        super().__init__()
        self.conv1 = GATv2Conv(num_node_features, hidden_dim, heads=heads, edge_dim=num_edge_features, concat=False)
        self.bn1 = BatchNorm1d(hidden_dim)
        
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, heads=heads, edge_dim=num_edge_features, concat=False)
        self.bn2 = BatchNorm1d(hidden_dim)
        
        self.conv3 = GATv2Conv(hidden_dim, hidden_dim, heads=heads, edge_dim=num_edge_features, concat=False)
        self.bn3 = BatchNorm1d(hidden_dim)
        
        jk_dim = hidden_dim * 3
        self.fc1 = Linear(jk_dim + fp_dim, hidden_dim * 2)
        self.drop = torch.nn.Dropout(0.5)
        self.fc2 = Linear(hidden_dim * 2, 5)

    def forward(self, data):
        x, edge_index, edge_attr, batch, fp = data.x, data.edge_index, data.edge_attr, data.batch, data.fp
        
        x1 = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x2 = F.relu(self.bn2(self.conv2(x1, edge_index, edge_attr)))
        x3 = F.relu(self.bn3(self.conv3(x2, edge_index, edge_attr)))
        
        # JUMPING KNOWLEDGE
        x_jk = torch.cat([x1, x2, x3], dim=-1)
        x_pool = global_mean_pool(x_jk, batch)
        
        x_out = torch.cat([x_pool, fp.squeeze(1)], dim=1)
        x_out = self.drop(F.relu(self.fc1(x_out)))
        return self.fc2(x_out)

In [5]:
# ==========================================
# 3. UTILS: SCAFFOLD SPLIT & MCC OPTIMIZATION
# ==========================================
def generate_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: 
        return ""
    
    #Explicitly strip all stereochemistry to prevent "bad bond stereo" crashes
    Chem.RemoveStereochemistry(mol)
    
    try:
        #Try-except block ensures a single bad molecule won't crash the whole pipeline
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold
    except Exception:
        # Fallback: If RDKit still fails, return an empty string to group it with other edge cases
        return ""

def scaffold_split(df, n_splits=5):
    scaffolds = defaultdict(list)
    for i, smiles in enumerate(df['SMILE']):
        scaffolds[generate_scaffold(smiles)].append(i)
        
    scaffold_sets = [idx_list for _, idx_list in sorted(scaffolds.items(), key=lambda x: len(x[1]), reverse=True)]
    folds = [[] for _ in range(n_splits)]
    fold_sizes = [0] * n_splits
    
    for scaffold_set in scaffold_sets:
        smallest_fold_idx = np.argmin(fold_sizes)
        folds[smallest_fold_idx].extend(scaffold_set)
        fold_sizes[smallest_fold_idx] += len(scaffold_set)
        
    all_indices = np.arange(len(df))
    return [(np.setdiff1d(all_indices, val_idx), val_idx) for val_idx in folds]

def find_best_thresholds(y_true, y_pred_probs):
    """Finds optimal threshold, but strictly clamped to prevent overfitting."""
    best_thresholds = []
    for i in range(5):
        best_thresh = 0.5
        best_mcc = -1
        # UNCLAMPED: Let the ensemble find its true high-confidence threshold
        for t in np.arange(0.30, 0.91, 0.02):
        # CHANGE HERE: Clamped range to prevent aggressive 0.25 thresholds
        #for t in np.arange(0.40, 0.61, 0.02): 
            preds = (y_pred_probs[:, i] > t).astype(int)
            mcc = matthews_corrcoef(y_true[:, i], preds)
            if mcc > best_mcc:
                best_mcc = mcc
                best_thresh = t
        best_thresholds.append(best_thresh)
    return best_thresholds

def calculate_mean_mcc(y_true, y_pred_probs, thresholds):
    mcc_scores = []
    for i in range(5):
        preds = (y_pred_probs[:, i] > thresholds[i]).astype(int)
        mcc_scores.append(matthews_corrcoef(y_true[:, i], preds))
    return np.mean(mcc_scores)

In [6]:
# ==========================================
# 4. TRAINING & INFERENCE PIPELINE
# ==========================================
def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            probs = torch.sigmoid(out).cpu().numpy()
            if probs.ndim == 1: probs = np.expand_dims(probs, axis=0)
            preds.extend(probs.tolist())
    return np.array(preds)

class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha, self.gamma, self.pos_weight = alpha, gamma, pos_weight
        
    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none', pos_weight=self.pos_weight)
        return (self.alpha * (1 - torch.exp(-bce)) ** self.gamma * bce).mean()
        
def train_cv(df_train, model_class, num_node_features, num_edge_features, n_splits=5, epochs=20): # Increased epochs to 20
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    y_train_full = df_train[['P1', 'P2', 'P3', 'P4', 'P5']].values
    
    num_positives = y_train_full.sum(axis=0)
    num_negatives = len(y_train_full) - num_positives
    pos_weights = torch.tensor(num_negatives / (num_positives + 1e-5), dtype=torch.float).to(device)
    
    oof_preds = np.zeros((len(df_train), 5))
    models = []
    splits = scaffold_split(df_train, n_splits=n_splits)

    for fold, (train_idx, val_idx) in enumerate(splits):
        # NEW: Added drop_last=True to prevent BatchNorm panics on small final batches
        train_loader = DataLoader(MoleculeDataset(df_train.iloc[train_idx]), batch_size=64, shuffle=True, drop_last=True)
        val_loader = DataLoader(MoleculeDataset(df_train.iloc[val_idx]), batch_size=64, shuffle=False)
        
        model = model_class(num_node_features, num_edge_features).to(device)
        optimizer = AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
        criterion = FocalLoss(gamma=2.0, pos_weight=pos_weights)
        
        for epoch in range(epochs):
            model.train()
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                
                # NEW: Removed .squeeze() so inputs/targets stay safely shaped as (Batch_Size, 5)
                loss = criterion(model(data), data.y) 
                
                loss.backward()
                optimizer.step()
            scheduler.step()
            
        oof_preds[val_idx] = predict(model, val_loader, device)
        models.append(model)
        
    return models, oof_preds

def generate_test_preds(models, df_test):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    loader = DataLoader(MoleculeDataset(df_test, is_test=True), batch_size=64, shuffle=False)
    return np.mean([predict(model, loader, device) for model in models], axis=0)

def to_rank_probs(preds):
    """Converts raw probabilities into uniform percentile ranks to stabilize the Meta-Model."""
    ranks = np.zeros_like(preds)
    for i in range(preds.shape[1]):
        # Ranks from 1 to N, then divide by N to get 0.0 -> 1.0 percentiles
        ranks[:, i] = rankdata(preds[:, i]) / len(preds[:, i])
    return ranks

In [7]:
# ==========================================
# 5. MAIN EXECUTION (Hybrid Tree-Graph Stacking)
# ==========================================
if __name__ == "__main__":
    # Adjust paths if running locally instead of Kaggle
    df_train = pd.read_csv('/kaggle/input/competitions/openaimer26-1/train.csv')
    df_test = pd.read_csv('/kaggle/input/competitions/openaimer26-1/test.csv')
    
    NUM_NODE_FEATURES = 9
    NUM_EDGE_FEATURES = 6

    # --- LEVEL 1: GRAPH MODELS ---
    print("1. Training GIN Models (Level 1)...")
    gin_models, gin_oof = train_cv(df_train, GINModel, NUM_NODE_FEATURES, NUM_EDGE_FEATURES, n_splits=8, epochs=30)
    
    print("2. Training GAT Models (Level 1)...")
    gat_models, gat_oof = train_cv(df_train, GATModel, NUM_NODE_FEATURES, NUM_EDGE_FEATURES, n_splits=8, epochs=30)

    # --- LEVEL 1: TABULAR MODELS ---
    print("3. Extracting Tabular Matrices...")
    X_train_fp, X_train_desc = extract_tabular(df_train)
    X_test_fp, X_test_desc = extract_tabular(df_test)
    X_train_tab = np.hstack((X_train_fp, X_train_desc))
    X_test_tab = np.hstack((X_test_fp, X_test_desc))
    y_train_multi = df_train[['P1', 'P2', 'P3', 'P4', 'P5']].values

    # --- LEVEL 1: TABULAR MODELS (UPGRADED) ---
    print("4. Training Tabular Baselines (XGB, RF, & Deep MLP)...")
    
    # Scale tabular data purely for the Neural Network
    scaler = StandardScaler()
    X_train_tab_scaled = scaler.fit_transform(X_train_tab)
    X_test_tab_scaled = scaler.transform(X_test_tab)
    
    # Calculate global imbalance weight to fix tree bias
    spw = np.mean((len(y_train_multi) - y_train_multi.sum(axis=0)) / (y_train_multi.sum(axis=0) + 1e-5))
    
    xgb_oof, rf_oof, mlp_oof = np.zeros((len(df_train), 5)), np.zeros((len(df_train), 5)), np.zeros((len(df_train), 5))
    xgb_models, rf_models, mlp_models = [], [], []
    splits = scaffold_split(df_train, n_splits=8) 
    
    for fold, (train_idx, val_idx) in enumerate(splits):
        # 1. Balanced XGBoost
        xgb = MultiOutputClassifier(XGBClassifier(n_estimators=400, learning_rate=0.02, max_depth=6, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw, n_jobs=-1, random_state=42))
        xgb.fit(X_train_tab[train_idx], y_train_multi[train_idx])
        xgb_oof[val_idx] = np.column_stack([p[:, 1] for p in xgb.predict_proba(X_train_tab[val_idx])])
        xgb_models.append(xgb)
        
        # 2. Balanced Random Forest
        rf = MultiOutputClassifier(RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_split=5, class_weight='balanced', n_jobs=-1, random_state=42))
        rf.fit(X_train_tab[train_idx], y_train_multi[train_idx])
        rf_oof[val_idx] = np.column_stack([p[:, 1] for p in rf.predict_proba(X_train_tab[val_idx])])
        rf_models.append(rf)
        
        # 3. NEW: Deep Tabular MLP
        # Three hidden layers to capture deep chemical interactions from the fingerprints
        mlp = MultiOutputClassifier(MLPClassifier(hidden_layer_sizes=(512, 256, 64), activation='relu', max_iter=200, alpha=1e-3, early_stopping=True, random_state=42))
        mlp.fit(X_train_tab_scaled[train_idx], y_train_multi[train_idx])
        mlp_oof[val_idx] = np.column_stack([p[:, 1] for p in mlp.predict_proba(X_train_tab_scaled[val_idx])])
        mlp_models.append(mlp)

    # --- LEVEL 2: META-MODEL STACKING ---
    print("5. Training Meta-Model Stack (Level 2)...")
    # Added mlp_oof to the Meta-Features matrix
    X_train_meta = np.hstack((gin_oof, gat_oof, xgb_oof, rf_oof, mlp_oof, X_train_desc))
    meta_models, oof_meta_preds = [], np.zeros((len(df_train), 5))
    
    for i in range(5): 
        meta_lr = LogisticRegression(max_iter=1500, C=0.05, class_weight='balanced')
        meta_lr.fit(X_train_meta, y_train_multi[:, i])
        oof_meta_preds[:, i] = meta_lr.predict_proba(X_train_meta)[:, 1]
        meta_models.append(meta_lr)

    # --- INFERENCE & OPTIMIZATION ---
    print("6. Generating Final Test Predictions...")
    gin_test_preds = generate_test_preds(gin_models, df_test)
    gat_test_preds = generate_test_preds(gat_models, df_test)
    xgb_test_preds = np.mean([np.column_stack([p[:, 1] for p in m.predict_proba(X_test_tab)]) for m in xgb_models], axis=0)
    rf_test_preds = np.mean([np.column_stack([p[:, 1] for p in m.predict_proba(X_test_tab)]) for m in rf_models], axis=0)
    
    # Generate Test Preds for the MLP using the SCALED test data
    mlp_test_preds = np.mean([np.column_stack([p[:, 1] for p in m.predict_proba(X_test_tab_scaled)]) for m in mlp_models], axis=0)
    
    # Add mlp_test_preds to the final test matrix
    X_test_meta = np.hstack((gin_test_preds, gat_test_preds, xgb_test_preds, rf_test_preds, mlp_test_preds, X_test_desc))
    
    final_test_preds = np.zeros((len(df_test), 5))
    for i in range(5):
        final_test_preds[:, i] = meta_models[i].predict_proba(X_test_meta)[:, 1]

    print("7. Optimizing MCC Thresholds & Formatting Submission...")
    optimal_thresholds = find_best_thresholds(y_train_multi, oof_meta_preds)
    print(f"Optimal Thresholds per property (P1-P5): {optimal_thresholds}")
    
    final_test_hard_labels = np.zeros_like(final_test_preds, dtype=int)
    for i in range(5):
        final_test_hard_labels[:, i] = (final_test_preds[:, i] > optimal_thresholds[i]).astype(int)
        
    submission = pd.DataFrame()
    submission['M-ID'] = df_test['M-ID']
    for i, prop in enumerate(['P1', 'P2', 'P3', 'P4', 'P5']):
        submission[prop] = final_test_hard_labels[:, i]

    submission.to_csv('submission.csv', index=False)
    print("JK-Ensemble Submission generated successfully!")

1. Training GIN Models (Level 1)...
2. Training GAT Models (Level 1)...
3. Extracting Tabular Matrices...
4. Training Tabular Baselines (XGB, RF, & Deep MLP)...
5. Training Meta-Model Stack (Level 2)...
6. Generating Final Test Predictions...
7. Optimizing MCC Thresholds & Formatting Submission...
Optimal Thresholds per property (P1-P5): [np.float64(0.6800000000000004), np.float64(0.6800000000000004), np.float64(0.7000000000000004), np.float64(0.7200000000000004), np.float64(0.7200000000000004)]
JK-Ensemble Submission generated successfully!
